# DefectVision AI - Metal Surface Defect Detection
## Phase 1: Train YOLOv8n on NEU-DET Dataset

This notebook trains a YOLOv8 nano model to detect 6 types of metal surface defects:
- Crazing
- Inclusion
- Patches
- Pitted surface
- Rolled-in scale
- Scratches

**Run this notebook in Google Colab with GPU runtime.**

1. Go to Runtime > Change runtime type > GPU (T4)
2. Run all cells
3. Download the trained weights at the end

## Step 1: Install Dependencies

In [ ]:
!pip install ultralytics roboflow -q

## Step 2: Check GPU Availability

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Step 3: Download NEU-DET Dataset from Roboflow

The NEU Surface Defect Dataset (NEU-DET) is from Northeastern University.
We use the Roboflow-hosted version which comes pre-formatted for YOLOv8.

**To get your free Roboflow API key:**
1. Go to [roboflow.com](https://roboflow.com) and create a free account
2. Go to Settings > API Keys
3. Copy your API key and paste it below

In [ ]:
from roboflow import Roboflow

# Replace with your Roboflow API key
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"  # <-- PASTE YOUR KEY HERE

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("harit-yadav-u3zph").project("neu-det-jkimb")
version = project.version(1)
dataset = version.download("yolov8")

print(f"\nDataset downloaded to: {dataset.location}")

## Step 4: Explore the Dataset

Let's look at what we downloaded -- the class distribution and some sample images.

In [ ]:
import os
import yaml
from pathlib import Path

# Read the data.yaml to see class info
data_yaml_path = os.path.join(dataset.location, "data.yaml")
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("Dataset configuration:")
print(f"  Classes: {data_config.get('names', 'N/A')}")
print(f"  Number of classes: {data_config.get('nc', 'N/A')}")

# Count images in each split
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(dataset.location, split, 'images')
    if os.path.exists(img_dir):
        count = len(os.listdir(img_dir))
        print(f"  {split}: {count} images")
    else:
        print(f"  {split}: directory not found")

In [ ]:
# Visualize some sample images
import matplotlib.pyplot as plt
import cv2
import random

train_img_dir = os.path.join(dataset.location, 'train', 'images')
sample_images = random.sample(os.listdir(train_img_dir), min(8, len(os.listdir(train_img_dir))))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, img_name in zip(axes.flatten(), sample_images):
    img = cv2.imread(os.path.join(train_img_dir, img_name))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(img_name[:20], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample Training Images - Metal Surface Defects', fontsize=14)
plt.tight_layout()
plt.show()

## Step 5: Train YOLOv8n Model

We use YOLOv8 **nano** (smallest variant) which:
- Has 3.2M parameters
- Fits easily on Colab's free T4 GPU
- Is fast enough for real-time inference
- Still achieves good accuracy on defect detection

Training for 50 epochs should take ~20-30 minutes on T4.

In [ ]:
from ultralytics import YOLO

# Load YOLOv8 nano pretrained on COCO (transfer learning)
model = YOLO("yolov8n.pt")

# Train on NEU-DET dataset
results = model.train(
    data=data_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    name="metal_defect_detector",
    patience=10,       # early stopping if no improvement for 10 epochs
    save=True,
    plots=True,        # generate training plots
)

## Step 6: Evaluate the Model

Check the model's performance on the validation set.

In [ ]:
# Validate on the validation set
metrics = model.val()

print(f"\n=== Metal Defect Detection Results ===")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

In [ ]:
# Show training curves
from IPython.display import Image, display

# Display training results
results_dir = Path("runs/detect/metal_defect_detector")

for plot_name in ["results.png", "confusion_matrix.png", "val_batch0_pred.png"]:
    plot_path = results_dir / plot_name
    if plot_path.exists():
        print(f"\n--- {plot_name} ---")
        display(Image(filename=str(plot_path), width=800))

## Step 7: Test on Sample Images

Run inference on validation images to see the detections.

In [ ]:
# Run inference on validation images
val_img_dir = os.path.join(dataset.location, 'valid', 'images')
if not os.path.exists(val_img_dir):
    val_img_dir = os.path.join(dataset.location, 'test', 'images')

val_images = os.listdir(val_img_dir)
sample_val = random.sample(val_images, min(6, len(val_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, img_name in zip(axes.flatten(), sample_val):
    img_path = os.path.join(val_img_dir, img_name)
    results = model(img_path, conf=0.25, verbose=False)
    annotated = results[0].plot()
    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated)
    ax.set_title(img_name[:25], fontsize=9)
    ax.axis('off')

plt.suptitle('YOLOv8n Predictions - Metal Surface Defects', fontsize=14)
plt.tight_layout()
plt.show()

## Step 8: Download Trained Weights

Download the `best.pt` file and place it in your local project at:
```
defect-vision/models/metal_yolo_best.pt
```

In [ ]:
import shutil

# Copy best weights to a convenient location
best_weights = results_dir / "weights" / "best.pt"
if best_weights.exists():
    shutil.copy(best_weights, "metal_yolo_best.pt")
    print(f"Best weights saved to: metal_yolo_best.pt")
    print(f"File size: {best_weights.stat().st_size / 1e6:.1f} MB")
else:
    # Try alternative path
    alt_path = Path("runs/detect/metal_defect_detector/weights/best.pt")
    if alt_path.exists():
        shutil.copy(alt_path, "metal_yolo_best.pt")
        print(f"Best weights saved to: metal_yolo_best.pt")
    else:
        print("Could not find best.pt - check the runs/ directory")

# Download to your local machine (works in Colab)
try:
    from google.colab import files
    files.download("metal_yolo_best.pt")
    print("\nDownload started! Save it to defect-vision/models/metal_yolo_best.pt")
except ImportError:
    print("Not running in Colab - copy metal_yolo_best.pt manually")